# 📈 IDX Bandarmology — End-to-End Notebook

This notebook runs the whole flow: **scrape → clean data → pipeline → analysis → modeling**,
to test the hypothesis:

> *Does "smart money" accumulation (big bandar / foreign brokers) in IDX stocks predict the next price increase?*

**Data sources:**
- `yfinance` — historical prices (OHLCV), free, no token needed.
- `Stockbit` (exodus.stockbit.com) — broker summary, bandar detector, foreign/domestic flow. Requires `STOCKBIT_TOKEN` in the `.env` file (see `.env.example`).

**How to use:** run the cells top to bottom. Edit `WATCHLIST` in the next cell to change which stocks are tracked.
Run this notebook every trading day (or schedule it) to accumulate more history — the more days you collect, the more reliable the correlation/modeling results become.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# repo root is the parent of this notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

from idx_bandarmology import config, pipeline, storage, features, analysis, modeling, stockbit

print("Stockbit token configured:", stockbit.is_available())
print("Default watchlist:", config.WATCHLIST)

## 2. Watchlist

Edit the list of stocks below. No `.JK` suffix needed — it's added automatically for yfinance
and stripped automatically for Stockbit.

In [ ]:
WATCHLIST = ["BBCA", "BBRI", "BMRI", "BBNI", "TLKM", "ASII", "UNVR", "GOTO", "BREN", "ANTM"]

# Or look up a new stock before adding it to the watchlist:
# from idx_bandarmology import stockbit
# stockbit.fetch_analysis("ADRO")["summary"]

## 3. Scrape data + run the pipeline

This single function does everything:
1. Fetch historical prices from yfinance for every ticker in the watchlist.
2. Fetch today's broker/bandar snapshot from Stockbit (if a token is available).
3. Store everything in SQLite (`data/db/bandarmology.sqlite`) — this is the "pipeline" landing zone that the Streamlit dashboard uses too.

Run this cell every trading day to accumulate broker/bandar history (Stockbit only provides a snapshot of the current day,
not a historical range — so the time series is built up from how many times you run this pipeline).

In [ ]:
result = pipeline.run(WATCHLIST, price_period="2y")

print(f"Prices: {result['n_prices']} rows")
print(f"Broker/bandar: {result['n_broker']} rows")

### 3a. Check the raw scrape per ticker (optional)

Useful for verifying Stockbit data before it enters the pipeline — each section (`broker`, `foreignDomestic`,
`pricePerformance`) is independent, so if one fails the others still show up.

In [ ]:
for sym, r in result["broker_results"].items():
    print("=" * 70)
    print(sym, "-", r.get("summary"))
    if r.get("broker", {}).get("available"):
        print(" ", r["broker"]["conclusion"])
    if r.get("foreignDomestic", {}).get("available"):
        print(" ", r["foreignDomestic"]["conclusion"])

## 4. Work with the data (pandas) — inspect the raw tables from the pipeline

Two main tables are stored in SQLite:
- `prices` — daily OHLCV per ticker (yfinance).
- `broker_flow` — broker/bandar snapshot per ticker per run day (Stockbit).

In [ ]:
price_df = storage.read_prices(WATCHLIST)
broker_df = storage.read_broker_flow(WATCHLIST)

print("prices:", price_df.shape)
display(price_df.tail())

print("\nbroker_flow:", broker_df.shape)
display(broker_df.tail())

## 5. Feature engineering

`features.build_feature_table()` joins `prices` + `broker_flow` into one tidy table,
and also computes the **forward return** (the return N days ahead from the day a signal appears) — this is the target
variable for the hypothesis test. We use forward return (not same-day) to avoid being circular:
we want to know whether *today's* signal predicts *future* price, not whether heavy buying
today mechanically pushes that same day's price up (which is always trivially true).

In [ ]:
feat = features.build_feature_table(WATCHLIST, horizons=(1, 3, 5, 10))
print(feat.shape)
feat.tail(10)[["ticker", "date", "close", "bandar_signal", "foreign_net_flow_pct",
               "fwd_return_1d", "fwd_return_5d", "fwd_return_10d"]]

## 6. Descriptive analysis & correlation

The core question: when today's bandar/foreign signal shows **accumulation**, does the price
actually rise N days later (compared to when the signal is **distribution** or neutral)?

In [ ]:
corr = analysis.correlation_table(feat)
display(corr)

In [ ]:
summary = analysis.summary_by_signal(feat, target_col="fwd_return_5d")
display(summary)

In [ ]:
fig = analysis.plot_signal_bucket_returns(feat, target_col="fwd_return_5d")
plt.show()

In [ ]:
fig = analysis.plot_signal_vs_forward_return(feat, horizon=5)
plt.show()

In [ ]:
# Correlation per ticker — the effect can differ from stock to stock
analysis.correlation_by_ticker(feat, target_col="fwd_return_5d")

In [ ]:
# Plot price + bandar signal for one ticker
fig = analysis.plot_price_with_signal(feat, WATCHLIST[0])
plt.show()

## 7. Modeling — statistical hypothesis test

Two approaches:

1. **Linear regression (OLS)** — is there a *statistically significant* relationship (p < 0.05)
   between the smart-money signals and the forward return, while controlling for other variables?
2. **Classification (Logistic Regression / Random Forest)** — if we reframe it as a binary question
   ("up or not"), how accurately do the smart-money signals predict it?

In [ ]:
reg = modeling.linear_regression(feat, target_col="fwd_return_5d")
print(f"n = {reg.n_obs}, R-squared = {reg.r_squared:.4f}")
display(reg.coefficients)

In [ ]:
# Full statsmodels summary (optional, more detail)
print(reg.summary_text)

In [ ]:
clf_logit = modeling.classify_up_down(feat, target_col="fwd_return_5d", model_type="logistic")
print(f"Logistic Regression — n={clf_logit.n_obs}, accuracy={clf_logit.accuracy:.1%}, "
      f"precision={clf_logit.precision:.1%}, recall={clf_logit.recall:.1%}, "
      f"ROC-AUC={clf_logit.roc_auc}")
display(clf_logit.feature_importance)

In [ ]:
clf_rf = modeling.classify_up_down(feat, target_col="fwd_return_5d", model_type="random_forest")
print(f"Random Forest — n={clf_rf.n_obs}, accuracy={clf_rf.accuracy:.1%}, "
      f"precision={clf_rf.precision:.1%}, recall={clf_rf.recall:.1%}, "
      f"ROC-AUC={clf_rf.roc_auc}")
display(clf_rf.feature_importance)

## 8. Verdict

In [ ]:
print(modeling.hypothesis_verdict(reg, clf_logit))

## 9. Dashboard

For interactive exploration (filter tickers, see all charts at once, re-run the pipeline
from the UI), open the Streamlit dashboard:

```bash
streamlit run dashboard/app.py
```

The dashboard reads from the same SQLite file as this notebook, so the data is always in sync — just
run the pipeline (from this notebook or from the button in the dashboard sidebar), then refresh.

> **Tip for posting to LinkedIn/GitHub:** screenshot the "Modeling / Hypothesis" tab in the dashboard, or export
> one of the charts above (`fig.savefig("chart.png", dpi=150)`) to use as a repo cover image.